## **OceanOSSE:** Development Notebook

### **Description:**

Notebook to develop & evaluate use of pyinterp Optimal Interpolation methods for regridding in OceanOSSE.

### **Created By:**

Ollie Tooth (oliver.tooth@noc.ac.uk)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pyinterp

### "Truth" Field

In [ ]:
def true_field(lon, lat, t_seconds):
    """Toy SLA: sinusoidal, slowly drifting eastward with time."""
    lon_phase = np.deg2rad(lon) - 1.0e-6 * t_seconds  # ~5 deg / day
    lat_phase = np.deg2rad(lat)
    return 0.2 * np.sin(3.0 * lon_phase) * np.cos(2.0 * lat_phase)

### Sparse Observations

In [ ]:
rng = np.random.default_rng(42)
n_obs = 600
lon = rng.uniform(-30.0, 0.0, n_obs)
lat = rng.uniform(30.0, 50.0, n_obs)
t = rng.uniform(0.0, 5 * 86400.0, n_obs)  # 5-day window

mission = rng.integers(0, 2, n_obs)
sigma_per_mission = np.array([0.02, 0.06])  # meters
sigma_obs = sigma_per_mission[mission]
obs_value = true_field(lon, lat, t) + rng.normal(0.0, sigma_obs)
obs_sigma2 = sigma_obs ** 2

In [ ]:
L_SPATIAL = 150e3        # metres
L_TIME = 7 * 86400.0     # seconds (7-day temporal decorrelation)
time_scale = L_SPATIAL / L_TIME  # ≈ 0.25 m/s

oi = pyinterp.OptimalInterpolation(
    np.column_stack([lon, lat, t]),
    obs_value,
    obs_sigma2,
    covariance="gaussian",
    coordinate_system="geographic",
    time_scale=time_scale,
)

### Analysis Grid

In [ ]:
nlon, nlat = 60, 60
glon = np.linspace(-30.0, 0.0, nlon)
glat = np.linspace(30.0, 50.0, nlat)
mlon, mlat = np.meshgrid(glon, glat, indexing="ij")
mt = np.full(mlon.size, 2.5 * 86400.0)
query = np.column_stack([mlon.ravel(), mlat.ravel(), mt])

### Uniform Length Scales

In [ ]:
res_uniform = oi(
    query,
    l_spatial=L_SPATIAL,    # metres
    lt=L_TIME,              # seconds
    sigma=0.20,             # metres
    k=24,
)
field_uniform = res_uniform.value.reshape(mlon.shape)
error_uniform = res_uniform.error.reshape(mlon.shape)

### Spatially-Varying Length Scales

In [ ]:
ax_lon = pyinterp.Axis(np.linspace(-30.0, 0.0, 11))
ax_lat = pyinterp.Axis(np.linspace(30.0, 50.0, 11))
ls_grid_array = np.linspace(200e3, 80e3, 11)[None, :] * np.ones((11, 11))
grid_ls = pyinterp.Grid2D(ax_lon, ax_lat, ls_grid_array)

res_varying = oi(
    query,
    l_spatial=grid_ls,
    lt=L_TIME,
    sigma=0.20,
    k=24,
)
field_varying = res_varying.value.reshape(mlon.shape)
error_varying = res_varying.error.reshape(mlon.shape)

### Visualisation

In [ ]:
truth = true_field(mlon, mlat, 2.5 * 86400.0)

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)

axes[0, 0].pcolormesh(mlon, mlat, truth, vmin=-0.2, vmax=0.2, cmap="RdBu_r")
axes[0, 0].set_title("Truth at t = 2.5 days")

axes[0, 1].pcolormesh(
    mlon, mlat, field_uniform, vmin=-0.2, vmax=0.2, cmap="RdBu_r"
)
axes[0, 1].set_title("OI — L_spatial = 150 km (uniform)")

axes[0, 2].pcolormesh(
    mlon, mlat, field_varying, vmin=-0.2, vmax=0.2, cmap="RdBu_r"
)
axes[0, 2].set_title("OI — L_spatial(lon, lat) from Grid2D")

near_t = np.abs(t - 2.5 * 86400.0) < 0.5 * 86400.0
axes[1, 0].scatter(
    lon[near_t], lat[near_t], c=obs_value[near_t],
    vmin=-0.2, vmax=0.2, cmap="RdBu_r", s=12
)
axes[1, 0].set_title(f"Observations within ±12h ({near_t.sum()} pts)")

axes[1, 1].pcolormesh(
    mlon, mlat, error_uniform, vmin=0.0, vmax=0.20, cmap="magma"
)
axes[1, 1].set_title("Formal error (m) — uniform L")

axes[1, 2].pcolormesh(
    mlon, mlat, error_varying, vmin=0.0, vmax=0.20, cmap="magma"
)
axes[1, 2].set_title("Formal error (m) — Grid2D L")

for ax in axes.flat:
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

fig.tight_layout()
plt.show()